# Comparing Groups

**DS4DH Practice Pack · Module 02 — Describing and Comparing Data**

*Technique:* Group means, absolute gaps, and reversal across levels of aggregation

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Sagaustus/ds4dh-colab-pack/blob/main/notebooks/02c_group_comparison.ipynb)

Data: `merged_dataset.csv`, `city_summary.csv` — from the `data/` folder of this pack.

---

In [ ]:
# Setup — run this first.
import os, warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd

# This notebook reads the CSVs sitting next to it. In Colab you will be asked
# to upload them from the pack's data/ folder.
NEEDED = ['merged_dataset.csv', 'city_summary.csv']

def _missing():
    return [f for f in NEEDED if not os.path.exists(f)]

missing = _missing()
if missing:
    try:
        from google.colab import files
    except ImportError:
        raise SystemExit('Place these next to the notebook: ' + ', '.join(missing))
    # Ask again until everything has arrived. The upload widget returns as soon
    # as you close it, so picking only some of the files would otherwise fail a
    # few lines below with a confusing FileNotFoundError.
    for _ in range(4):
        print('Select ALL of these at once (ctrl-click / cmd-click to multi-select):')
        print('   ' + ', '.join(missing))
        files.upload()
        missing = _missing()
        if not missing:
            break
        print('Still needed: ' + ', '.join(missing))
    if missing:
        raise SystemExit(
            'Missing: ' + ', '.join(missing) + '. Re-run this cell and select '
            'every file listed, or upload them with the folder icon on the left.')

df       = pd.read_csv('merged_dataset.csv')
df_city  = pd.read_csv('city_summary.csv')
CITIES = ['Montréal', 'Toronto', 'Edmonton', 'Vancouver']

print(f'Loaded. df has {len(df):,} rows and {df.shape[1]} columns.')

## What this notebook does

Comparison is where an analysis starts making claims, and where it starts being
able to be wrong.

This notebook builds the immigrant/non-immigrant comparison two ways — once by
comparing city averages, once by comparing within each CSD — and shows that they
can disagree. Deciding which is right is not a statistical question.

In [ ]:
# Both groups, at CSD level.
csd = df.dropna(subset=['csd_code'])
g = csd[csd['immigrant_status'].isin(['Immigrant', 'Non-immigrants'])
        & csd['cma'].isin(CITIES)]

print(f'{"City":<12}{"group":<18}{"n":>5}{"mean Renter":>13}{"median":>9}')
print('-' * 57)
for city in CITIES:
    for grp in ['Immigrant', 'Non-immigrants']:
        s = g[(g['cma'] == city) & (g['immigrant_status'] == grp)]['Renter'].dropna()
        print(f'{city:<12}{grp:<18}{len(s):>5}{s.mean():>13.2f}{s.median():>9.2f}')
    print()

## Method A — difference of city means

Take each group's mean within a city and subtract. Simple, and it is what the
pre-computed `city_summary.csv` does.

In [ ]:
print('Method A — difference of group means within each city')
print()
print(f'{"City":<12}{"immigrant":>11}{"non-imm":>10}{"gap (pp)":>11}')
print('-' * 44)
method_a = {}
for city in CITIES:
    i = g[(g['cma'] == city) & (g['immigrant_status'] == 'Immigrant')]['Renter'].mean()
    n = g[(g['cma'] == city) & (g['immigrant_status'] == 'Non-immigrants')]['Renter'].mean()
    method_a[city] = i - n
    print(f'{city:<12}{i:>11.2f}{n:>10.2f}{i - n:>+11.2f}')

## Method B — mean of within-CSD differences

Pair the two groups *inside each municipality*, take the difference there, then
average those differences. This holds the place constant — a CSD's own housing
market cannot contaminate the comparison.

In [ ]:
imm = csd[csd['immigrant_status'] == 'Immigrant'][['csd_code', 'cma', 'Renter']]
nim = csd[csd['immigrant_status'] == 'Non-immigrants'][['csd_code', 'Renter']]
pair = imm.merge(nim, on='csd_code', suffixes=('_imm', '_nim')).dropna()
pair = pair[pair['cma'].isin(CITIES)]
pair['gap'] = pair['Renter_imm'] - pair['Renter_nim']

print('Method B — mean of paired within-CSD differences')
print()
print(f'{"City":<12}{"n pairs":>9}{"gap (pp)":>11}{"Method A":>11}{"differ by":>11}')
print('-' * 54)
for city in CITIES:
    b = pair[pair['cma'] == city]['gap'].mean()
    a = method_a[city]
    print(f'{city:<12}{(pair["cma"] == city).sum():>9}{b:>+11.2f}{a:>+11.2f}{b - a:>+11.2f}')

The two methods do not agree, and they cannot be reconciled by being more careful.

Method A averages over *different sets of CSDs* for each group, because
suppression is not identical between them. Method B restricts to CSDs where both
groups are reported — a smaller, more urban sample, but a genuine like-for-like
comparison.

**Method B is the defensible one for a gap claim.** Method A answers "how do the
published averages compare"; Method B answers "within the same place, do these
two groups differ".

### 🔧 Your turn 1

Print `len(pair)` and compare it to the number of CSDs with any renter data.

How much of the dataset did Method B discard to get its like-for-like comparison?
Is that trade worth it?

In [ ]:
# Which places drive the gap? Rank by absolute difference.
ranked = pair.reindex(pair['gap'].abs().sort_values(ascending=False).index)
top = ranked.head(8).merge(
    csd[['csd_code', 'geography_name']].drop_duplicates('csd_code'), on='csd_code')
print(top[['geography_name', 'cma', 'Renter_imm', 'Renter_nim', 'gap']]
      .to_string(index=False))

### 🔧 Your turn 2

Look at the `n pairs` column from Method B against the size of the gaps.

Which city has both the largest gap and the fewest pairs? What does that
combination usually mean, and what would you do before quoting its number?

<details markdown="1">
<summary><b>What you should have seen</b> — click to expand</summary>

**Your turn 1.** Method B keeps 132 CSDs out of the ~189 in the four cities.
Roughly 30% of places are discarded because one of the two groups is suppressed
there. That is a real cost, and it biases the remaining sample toward larger,
more urban municipalities. It is still the right trade: a comparison across
non-matching sets of places is not a comparison of groups at all, it is a
comparison of groups *and* places, with no way to separate the two.

**Your turn 2.** Edmonton — the largest mean gap (about −3.2pp) on only 13 pairs.
A large effect on a small sample is the classic signature of either a genuine
strong effect or a couple of unusual observations. Before quoting it you would
test whether it is distinguishable from zero and compute an effect size. That is
exactly what notebooks 04a–04c do, and Edmonton turns out to survive — which is
the surprise of this course.

</details>

## Where this stops

You have a gap, computed defensibly. You have no basis yet for saying it is
anything other than the particular CSDs that happened to be reported. Module 04
supplies that basis.